# Decision Tree Classification: Model Comparison, ROC Curves, and XGBoost
In this notebook, we will walk through building a Decision Tree classifier
using the Breast Cancer Wisconsin (Diagnostic) dataset. Decision trees are non-parametric models that
can capture nonlinear relationships by recursively splitting the data.
____
We'll cover:
1. Loading and inspecting the dataset.
2. Preprocessing: Handling missing values and encoding categorical data.
3. Splitting the data into training and testing sets.
4. Training a Decision Tree model.
5. Evaluating the model's performance with accuracy, confusion matrix, and a classification report.
6. Visualizing the decision tree.
7. Analyzing model performance using ROC curve and AUC.
8. Comparing Decision Tree, Logistic Regression, Random Forest, and XGBoost via ROC curves.

## **Step 1: Load and Inspect the Data**

We use the Breast Cancer Wisconsin (Diagnostic) dataset from Kaggle/UCI, which includes features computed from digitized images of fine needle aspirates (FNA) of breast masses. Each row describes characteristics of cell nuclei (e.g., radius, texture, smoothness), and the target variable indicates whether the tumor is **benign (B)** or **malignant (M)**.

In [5]:
# Importing necessary libraries
import numpy as np
import pandas as pd
import seaborn as sns
import kagglehub

# Download the Breast Cancer Wisconsin dataset from Kaggle
path = kagglehub.dataset_download("uciml/breast-cancer-wisconsin-data")

print("Path to dataset files:", path)

data = pd.read_csv(f"{path}/data.csv")
data.head()

ModuleNotFoundError: No module named 'kagglehub'

### **Step 2: Data Preprocessing**

Our next step is to prepare the data for modeling:

- **Handling Missing Values:**
   The breast cancer dataset is mostly complete, but it contains an unnamed trailing column (`Unnamed: 32`) that is all NaN. We will drop it along with the `id` column, which is not a predictive feature.

- **Encoding Categorical Variables:**
   The `diagnosis` column contains categorical values (`M` for malignant, `B` for benign). We convert this to a numeric format using one-hot encoding.

*Note: We use `drop_first=True` to avoid the dummy variable trap, so `diagnosis_M` = 1 means malignant and 0 means benign.*

In [ ]:
data.info()

In [ ]:
# Handling missing values (optional for decision trees)
#data.dropna(inplace=True) # May be needed for logistic regression

# Encode the 'diagnosis' column: M (malignant) -> 1, B (benign) -> 0
df = pd.get_dummies(data, columns=['diagnosis'], drop_first=True) # Use drop_first=True to avoid "dummy trap"


# Define features (X) and target (y)
# Drop non-predictive columns: id, Unnamed: 32 (empty), and the target column
X = df.drop(columns=["id", "Unnamed: 32", "diagnosis_M"])
y = df['diagnosis_M']

# Preview the cleaned dataset
print(X.head())
print(y.head())

### **Step 3: Splitting the Data**

We split the dataset into training and testing sets. The training set is used to build the decision tree model, while the testing set is used to evaluate its performance.

In [ ]:
from sklearn.model_selection import train_test_split

# Split dataset into training and testing subsets
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=42)

### **Step 4: Train the Decision Tree Model**

We initialize and train a Decision Tree classifier.
**Why Decision Trees?**
- They are intuitive and easy to interpret.
- They capture non-linear relationships without needing feature scaling.

Here, we use default parameters, but tuning (e.g., max_depth, min_samples_split) can improve performance and prevent overfitting.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Initialize and train tree classification model
model = DecisionTreeClassifier(random_state = 42,
                               max_depth = 4) # change default parameters
model.fit(X_train, y_train)

### **Step 5: Evaluate the Model**

We now assess our model’s performance on the test data using several metrics:

- **Accuracy:** The overall proportion of correct predictions.
- **Confusion Matrix:** Displays the number of correct and incorrect predictions.
- **Classification Report:** Provides precision, recall, and F1-score, which help in understanding performance per class.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Predict on test data
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Display classification report
print(classification_report(y_test, y_pred))

### **Step 6: Visualizing the Decision Tree**

One of the advantages of decision trees is their interpretability. We can visualize the tree structure using the graphviz library.
The visualization shows:
- Splitting criteria at each node.
- Feature names used for splits.
- Class distributions within the nodes.

In [ ]:
# Import graphviz and export the decision tree to dot format for visualization
import graphviz
from sklearn import tree  # Ensure to import the tree module from sklearn

dot_data = tree.export_graphviz(model, feature_names=X_train.columns,
                                class_names=["Benign", "Malignant"],
                                filled=True)

# Generate and display the decision tree graph
graph = graphviz.Source(dot_data)
graph

### **Step 7: ROC Curve and AUC Analysis**
The ROC (Receiver Operating Characteristic) curve helps evaluate the model's
performance across different classification thresholds:

- **ROC Curve:** Plots True Positive Rate (TPR) against False Positive Rate (FPR).
- **AUC (Area Under the Curve):** Summarizes the overall ability of the model to discriminate between classes.

Here, we calculate and plot the ROC curve along with the AUC score.

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# Get the predicted probabilities for the positive class (malignant)
y_probs = model.predict_proba(X_test)[:, 1]

# Calculate the False Positive Rate (FPR), True Positive Rate (TPR), and thresholds
fpr, tpr, thresholds = roc_curve(y_test, y_probs)

# Compute the Area Under the Curve (AUC) score
roc_auc = roc_auc_score(y_test, y_probs)
print(f"ROC AUC Score: {roc_auc:.2f}")

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, lw=2, label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], lw=2, linestyle='--', label='Random Guess') # Plotting 50% line
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

### **Step 8: Train a Logistic Regression Model**

To compare against our Decision Tree, we now train a Logistic Regression model on the same data. Logistic Regression is a linear model that estimates the probability of a binary outcome, making it a natural baseline for classification tasks like this one.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Initialize and train logistic regression model
lr_model = LogisticRegression()
lr_model.fit(X_train, y_train)

# Predict on test data
y_pred = lr_model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Display classification report
print(classification_report(y_test, y_pred))

In [ ]:
# Get the predicted probabilities for the positive class (malignant)
y_probs_lr = lr_model.predict_proba(X_test)[:, 1]

# Calculate the False Positive Rate (FPR), True Positive Rate (TPR), and thresholds
fpr_lr, tpr_lr, thresholds_lr = roc_curve(y_test, y_probs_lr)

# Compute the Area Under the Curve (AUC) score
roc_auc_lr = roc_auc_score(y_test, y_probs_lr)
print(f"ROC AUC Score: {roc_auc_lr:.2f}")

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, lw=2, label=f'DT ROC Curve (AUC = {roc_auc:.2f})')
plt.plot(fpr_lr, tpr_lr, lw=2, label=f'LR ROC Curve (AUC = {roc_auc_lr:.2f})')
plt.plot([0, 1], [0, 1], lw=2, linestyle='--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

### **Step 9: Train a Random Forest Model**

Random Forest is an ensemble method that builds multiple decision trees on random subsets of the data and averages their predictions. This reduces overfitting compared to a single decision tree and typically improves generalization. We compare its ROC curve against the Decision Tree and Logistic Regression models.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
random_forest_classifier = RandomForestClassifier(random_state=42)
random_forest_classifier.fit(X_train, y_train)

# Predict on test data
y_pred = random_forest_classifier.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Display classification report
print(classification_report(y_test, y_pred))

In [ ]:
# Get the predicted probabilities for the positive class (malignant)
y_probs_rf = random_forest_classifier.predict_proba(X_test)[:, 1]

# Calculate the False Positive Rate (FPR), True Positive Rate (TPR), and thresholds
fpr_rf, tpr_rf, thresholds_rf = roc_curve(y_test, y_probs_rf)

# Compute the Area Under the Curve (AUC) score
roc_auc_rf = roc_auc_score(y_test, y_probs_rf)
print(f"ROC AUC Score: {roc_auc_rf:.2f}")

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, lw=2, label=f'DT ROC Curve (AUC = {roc_auc:.2f})')
plt.plot(fpr_lr, tpr_lr, lw=2, label=f'LR ROC Curve (AUC = {roc_auc_lr:.2f})')
plt.plot(fpr_rf, tpr_rf, lw=2, label=f'RF ROC Curve (AUC = {roc_auc_rf:.2f})')
plt.plot([0, 1], [0, 1], lw=2, linestyle='--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()

### **Step 10: Train an XGBoost Model**

XGBoost (Extreme Gradient Boosting) is a powerful ensemble method that builds trees sequentially, where each new tree corrects the errors of the previous ones. It often achieves top performance on structured/tabular data and is widely used in machine learning competitions and industry applications.

In [ ]:
from xgboost import XGBClassifier

# Initialize and train XGBoost model
xg_model = XGBClassifier()
xg_model.fit(X_train, y_train)

# Predict on test data
y_pred = xg_model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

# Display classification report
print(classification_report(y_test, y_pred))

In [ ]:
# Get the predicted probabilities for the positive class (malignant)
y_probs_xg = xg_model.predict_proba(X_test)[:, 1]

# Calculate the False Positive Rate (FPR), True Positive Rate (TPR), and thresholds
fpr_xg, tpr_xg, thresholds_xg = roc_curve(y_test, y_probs_xg)

# Compute the Area Under the Curve (AUC) score
roc_auc_xg = roc_auc_score(y_test, y_probs_xg)
print(f"ROC AUC Score: {roc_auc_xg:.2f}")

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, lw=2, label=f'DT ROC Curve (AUC = {roc_auc:.2f})')
plt.plot(fpr_lr, tpr_lr, lw=2, label=f'LR ROC Curve (AUC = {roc_auc_lr:.2f})')
plt.plot(fpr_xg, tpr_xg, lw=2, label=f'XG ROC Curve (AUC = {roc_auc_xg:.2f})')
plt.plot(fpr_rf, tpr_rf, lw=2, label=f'RF ROC Curve (AUC = {roc_auc_rf:.2f})')
plt.plot([0, 1], [0, 1], lw=2, linestyle='--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.show()